In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

### P1_P2 (Day 4) (10 points, coding task)




**The high level idea of PINN is as follows:**

1. **(Neural network)** We design a neural network (functional mapping) $U(\cdot, \cdot | \theta): [0,1]^2 \to \mathbb{R}$, such that
   - $\theta$: learnable parameters in $U$.
   - Inputs are time $t$ and position $x$.
   - Output is the predicted temperature.

2. **(Training data)** To train $U$ (equivalently, to learn $\theta$), we use the following three groups of temporal-spacial data $(t, x)$:
   - **(Training data for PDE)** $(t, x)$ are randomly sampled from $[0,1]^2$. Denote by $\mathcal{D}_{PDE}$ the set of these data points.
   - **(Training data for IC)** $(0, x)$ with $x$ that are evenly distributed on $[0, 1]$. Denote by $\mathcal{D}_{IC}$ the set of these data points.
   - **(Training data for BC)** $(t, 0)$ and $(t, 1)$ with $t$ that are evenly distributed on $[0, 1]$. Denote by $\mathcal{D}_{BC}$ the set of these data points.

3. **(Loss function in training)**
   $$L_{total} = L_{PDE} + L_{IC} + L_{BC},$$
   where
   - Residual loss in PDE:
     $$L_{PDE} = \frac{1}{|\mathcal{D}_{PDE}|} \sum_{(t,x)\in \mathcal{D}_{PDE}} \left( \frac{\partial U(t,x|\theta)}{\partial t} - \alpha \frac{\partial^2 U(t,x|\theta)}{\partial x^2} \right)^2.$$
   - IC loss:
     $$L_{IC} = \frac{1}{|\mathcal{D}_{IC}|} \sum_{(t,x)\in \mathcal{D}_{IC}} (U(t,x|\theta) - u(t,x))^2.$$
   - BC loss:
     $$L_{BC} = \frac{1}{|\mathcal{D}_{BC}|} \sum_{(t,x)\in \mathcal{D}_{BC}} (U(t,x|\theta) - u(t,x))^2.$$

---

In this part, you are asked to build a deep neural network that is used to output PDE solutions.

1. The class name is `HeatPINN`. It subclasses `nn.Module`.

2. The model consists of the following layers that are sequentially connected:
   (1) Fully connected layer with `out_features = 64` (you need to determine `in_features` taken from the input).
   (2) Activation layer with `tanh` function.
   (3) Fully connected layer with `in_features = 64` and `out_features = 64`.
   (4) Activation layer with `tanh` function.
   (5) Fully connected layer with `in_features = 64` (you need to determine `out_features` as the output of the entire model).

3. Construct a model who is an object of this class and is called as `model`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class HeatPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = HeatPINN()

""" END OF THIS PART """


### P1_P3 (Day 4) (5 points, coding task)

Do the following tasks:

Let $x$ be a tensor with shape $(N,2)$.

1. What is the number of dimensions of `model(x)`?
2. What is the shape of `model(x)`?
3. Explain the reasoning of your answers.

In [ ]:
# 1. The dimension of model(x) is 2.
# 2. The shape of model(x) is (N,1).
# 3. For each sample, the output is a 1-dim tensor with shape (1,). Therefore, by having N samples, we get the answers above.

### P1_P4 (Day 4) (10 points, coding task)

In this part, you are asked to create the dataset $\mathcal{D}_{PDE}$.

1. The dataset object is called `dataset_train_PDE`. It is in a class called `Dataset_PDE` that you need to build.
2. Class `Dataset_PDE` subclasses `Dataset`.
3. Each $(t,x) \in \mathcal{D}_{PDE}$ is randomly sampled from $[0,1]^2$.
4. Set $|\mathcal{D}_{PDE}| = 500$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

num_samples = 500

class Dataset_PDE(Dataset):
    def __init__(self, num_samples):
        self.data = torch.rand(num_samples, 2, requires_grad=True)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]

dataset_train_PDE = Dataset_PDE(num_samples)

### P1_P5 (Day 4) (5 points, coding task)

In this part, you are asked to define a Dataloader object called `dataloader_PDE`:

1. `dataset = dataset_train_PDE`
2. `batch_size = 32`
3. `shuffle = True`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

batch_size_PDE = 32

dataloader_PDE = DataLoader(dataset_train_PDE, batch_size=batch_size_PDE, shuffle=True)

### P1_P6 (Day 3) (10 points, coding task)

In this part, you are asked to create the dataset $\mathcal{D}_{IC}$.

1. Define dataset $\mathcal{D}_{IC}$ in the way that for each $(t, x) \in \mathcal{D}_{IC}$, $t$ is fixed at 0 and $x$ is evenly sampled from $\{0, 0.01, 0.02, \dots, 0.98, 0.99, 1\}$. Therefore, $|\mathcal{D}_{IC}| = 101$.

2. The dataset shall be a tensor with name `dataset_train_IC` and shape `(101, 2)`.

3. Set `dataset_train_IC.requires_grad = True`.

4. Print `dataset_train_IC.requires_grad` and `dataset_train_IC.shape`.

5. Define tensor `u_IC` to be the ground-truth functional values of all data in $\mathcal{D}_{IC}$ (You can find the formula from Part 1).

6. Set `u_IC.requires_grad = True` and `u_IC.shape = (101, 1)`.

7. Print `u_IC.requires_grad` and `u_IC.shape`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

num_samples = 101

dataset_train_IC = torch.stack([torch.zeros(num_samples), torch.linspace(0, 1, num_samples)], dim=1)
dataset_train_IC.requires_grad = True

print(dataset_train_IC.requires_grad)
print(dataset_train_IC.shape)

u_IC = torch.sin(torch.pi * dataset_train_IC[:, 1].view(-1, 1))
u_IC.requires_grad_(True)

print(u_IC.requires_grad)
print(u_IC.shape)

### P1_P7 (Day 2) (10 points, co|ding task)

In this part, you are asked to create the dataset $\mathcal{D}_{BC}$.

1. Define dataset $\mathcal{D}_{BC}$ in the way that for each $(t, x) \in \mathcal{D}_{BC}$, $x$ is either 0 or 1, and $t$ is evenly sampled from $\{0, 0.01, 0.02, \dots, 0.98, 0.99, 1\}$. Therefore, $|\mathcal{D}_{BC}| = 2 \cdot 101 = 202$.

2. The dataset shall be a tensor with name `dataset_train_BC` and shape `(202, 2)`.

3. Set `dataset_train_BC.requires_grad = True`.

4. Print `dataset_train_BC.requires_grad` and `dataset_train_BC.shape`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

num_samples = 202

data_0 = torch.stack([torch.linspace(0, 1, num_samples//2), torch.zeros(num_samples//2)], dim=1)
data_1 = torch.stack([torch.linspace(0, 1, num_samples//2), torch.ones(num_samples//2)], dim=1)
dataset_train_BC = torch.cat([data_0, data_1], dim=0)
dataset_train_BC.requires_grad_(True)

print(dataset_train_BC.requires_grad)
print(dataset_train_BC.shape)

### P1_P8 (Day 4) (5 points, coding task)

In this part, you are asked to configure your optimizer.

1. Define an optimizer object called `optimizer`.
2. Configure the optimization method as `Adam`.
3. Set the learning rate as `1e-3`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=lr)

### P1_P9 (Day 3) (10 points, coding task)

The purpose of this part is to guide you to learn using `torch.autograd.grad`.

For each given input $(t, x)$, we not only compute $U(t, x | \theta)$ (output from the model), but also its 1st and 2nd order partial derivatives.

In PyTorch, these can be done by using `torch.autograd.grad`.

**Part 9.1**

Consider the following function
$$ f(p, q) = p^2 + q^3 + pq^2 .$$

**Do the following tasks at $(p, q) = (1, 2)$**:

1. Define tensors `p` and `q` that have values 1.0 and 2.0 (float data type), respectively, an identical shape `()` (that is, 0-dim), and `requires_grad = True`.

2. Compute tensor `f` according to the formula above.

3. Compute $\frac{\partial f(p,q)}{\partial p}$ by using
   ```python
   f_p = autograd.grad(f, p, create_graph=True)[0]
   ```
   Print `f_p`.

4. Compute $\frac{\partial f(p,q)}{\partial q}$ by using
   ```python
   f_q = autograd.grad(f, q, create_graph=True)[0]
   ```
   Print `f_q`.

5. Compute $\frac{\partial^2 f(p,q)}{\partial p^2}$ by using
   ```python
   f_pp = autograd.grad(f_p, p, create_graph=True)[0]
   ```
   Print `f_pp`.

6. Compute $\frac{\partial^2 f(p,q)}{\partial q^2}$ by using
   ```python
   f_qq = autograd.grad(f_q, q, create_graph=True)[0]
   ```
   Print `f_qq`.

7. Compute $\frac{\partial^2 f(p,q)}{\partial p \partial q}$ by using
   ```python
   f_pq = autograd.grad(f_p, q, create_graph=True)[0]
   ```
   Print `f_pq`.

8. Compute $\frac{\partial^3 f(p,q)}{\partial q^3}$ by using
   ```python
   f_qqq = autograd.grad(f_qq, q, create_graph=True)[0]
   ```
   Print `f_qqq`.

**Part 9.2**

Consider the following function
$$ g(x) = x^2 .$$

Let $x$ be a vector with values $0, 0.1, \dots, 0.9, 1$.

**Do the following tasks.**

1. Generate `x` as a 1-dim tensor and set `x.requires_grad = True`.

2. Generate `g = x**2`. Thus, `g` has the same shape as `x`.

3. Define `g_x` to be an element-wise 1st-order derivative of function $g$ with respect to $x$. Thus, `g_x` has the same shape as `x`.

   Write code to compute `g_x`. Print `g_x` and `g_x.shape`.

   Hint: by using `autograd.grad(f, x, create_graph=True)[0]`, tensor `x` can be with any dimension, but tensor `f` must be with dimension 0. In this problem, tensor `g` is not with dimension 0. So you need to think about how to address this issue.

4. Define `g_xx` to be an element-wise 2st-order derivative of function $g$ with respect to $x$. Thus, `g_xx` has the same shape as `x`. Write code to compute `g_xx`. Print it out `g_xx` and `g_xx.shape`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###
import torch.autograd as autograd

# Part 9.1
p = torch.tensor(1.0, requires_grad=True)
q = torch.tensor(2.0, requires_grad=True)

f = p**2 + q**3 + p*q**2

f_p = autograd.grad(f, p, create_graph=True)[0]
print(f_p)

f_q = autograd.grad(f, q, create_graph=True)[0]
print(f_q)

f_pp = autograd.grad(f_p, p, create_graph=True)[0]
print(f_pp)

f_qq = autograd.grad(f_q, q, create_graph=True)[0]
print(f_qq)

f_pq = autograd.grad(f_p, q, create_graph=True)[0]
print(f_pq)

f_qqq = autograd.grad(f_qq, q, create_graph=True)[0]
print(f_qqq)

# Part 9.2

x = torch.linspace(0, 1, 11)
x.requires_grad_(True)

g = x**2
# To compute gradients for non-scalar outputs, we can sum the output or provide gradients argument.
# The hint suggests sum: "tensor f must be with dimension 0" implies we should reduce g to a scalar.
g_x = autograd.grad(torch.sum(g), x, create_graph=True)[0]
print(g_x)
print(g_x.shape)

g_xx = autograd.grad(torch.sum(g_x), x, create_graph=True)[0]
print(g_xx)
print(g_xx.shape)

### P1_P10 (Day 4) (10 points, coding task)

This part asks you to do a mini-batch training of the model.

1. Set the parameter in the PDE `alpha = 0.1`. (This is not for learning. In PINN, we know the exact form of a PDE. We just need neural networks to help us solve it.)

2. Set the number of epochs as 1000.

3. Define
   ```python
   device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
   ```

4. While iterating over epochs, use `tqdm` to track the progress:
   ```python
   for epoch in tqdm(range(num_epochs)):
   ```
   
5. In each epoch,

   (1) Configure the model to the training mode.

   (2) Iterate over all mini-batches of `dataset_train_PDE`.

   (3) For each of the above mini-batch of the PDE dataset, while computing the total loss function, you also need to use **all** data in `dataset_train_IC` and `dataset_train_BC`.

   (4) Do all these tasks on **GPU**.

6. In each epoch, after training over all mini-batches, if the epoch index is divisible by 100, do the following tasks:

   (1) Configure the model to the evaluation mode.

   (2) Compute the total loss over the entire three datasets: `dataset_train_PDE`, `dataset_train_IC`, `dataset_train_BC`.

   (3) Print the epoch index, the residual loss from PDE, the IC loss, the BC loss, and the total loss.

   (4) Do all these tasks on **CPU**.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

alpha = 0.1
num_epochs = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for epoch in tqdm(range(num_epochs)):
    model.train()
    model.to(device)
    for batch_PDE in dataloader_PDE:
        batch_PDE = batch_PDE.to(device)
        optimizer.zero_grad()
        U_PDE = model(batch_PDE)
        U_t_and_x_PDE = autograd.grad(torch.sum(U_PDE), batch_PDE, create_graph=True)[0]
        U_t_PDE = U_t_and_x_PDE[:,0]
        U_x_PDE = U_t_and_x_PDE[:,1]
        U_xx_PDE = autograd.grad(torch.sum(U_x_PDE), batch_PDE, create_graph=True)[0][:,1]
        loss_PDE = torch.mean((U_t_PDE - alpha * U_xx_PDE)**2)

        U_IC = model(dataset_train_IC.to(device))
        loss_IC = torch.mean((U_IC - u_IC.to(device))**2)

        U_BC = model(dataset_train_BC.to(device))
        loss_BC = torch.mean(U_BC**2)
        
        loss = loss_PDE + loss_IC + loss_BC
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0:
        model.eval()
        model.to('cpu')
        batch_PDE = next(iter(dataloader_PDE))
        batch_PDE.requires_grad_(True)
        U_PDE = model(batch_PDE)
        U_t_and_x_PDE = autograd.grad(torch.sum(U_PDE), batch_PDE, create_graph=True)[0]
        U_xx_PDE = autograd.grad(torch.sum(U_t_and_x_PDE[:,1]), batch_PDE, create_graph=True)[0][:,1]
        loss_PDE = torch.mean((U_t_and_x_PDE[:,0] - alpha * U_xx_PDE)**2)

        with torch.no_grad():
            U_IC = model(dataset_train_IC)
            loss_IC = torch.mean((U_IC - u_IC)**2)
            U_BC = model(dataset_train_BC)
            loss_BC = torch.mean(model(dataset_train_BC)**2)
            loss = loss_PDE + loss_IC + loss_BC

        print(f"{loss_PDE.item():.4e}, {loss_IC.item():.4e}, {loss_BC.item():.4e}")
        print(f"Epoch {epoch}, Loss: {loss.item():.4e}")

### P2_P1 (Day 8) (5 points, non-coding task)

**Multi-head attention (MHA)** is a big breakthrough in AI. Based on its original form, there are many variants that improved it.

In this problem, you are asked to study multi-head attention and its variants.

We use the following notation in this problem.

- $B$: batch size. $b$: index of a sample.

- $L_1$: length of an attending sequence. $l_1$: index of a position in this sequence.

- $L_2$: length of a being attended sequence. $l_2$: index of a position in this sequence.

- $D_1$: dimension of a hidden state/token in an attending sequence.

- $D_2$: dimension of a hidden state/token in a being attended sequence.

- $H$: number of heads. $h$: index of a head.

- $D_v$: dimension of a value vector.

- $D_{qk}$: dimension of a query/key vector.


**Do the following tasks (Reasoning is not required).**

1. For each hidden state at position $l_1$ in an attending sequence, $\mathbf{x}_{l_1} \in \mathbb{R}^{D_1}$, we project it into a query vector for head $h$ according to
   $$ \mathbf{q}_{l_1, h} = \mathbf{W}_h^Q \mathbf{x}_{l_1}. $$
   What is the shape of $\mathbf{W}_h^Q$?

2. For each hidden state at position $l_2$ in a being attended sequence $\mathbf{y}_{l_2} \in \mathbb{R}^{D_2}$, we project it into a key vector for head $h$ according to
   $$ \mathbf{k}_{l_2, h} = \mathbf{W}_h^K \mathbf{y}_{l_2}. $$
   What is the shape of $\mathbf{W}_h^K$?

3. For each hidden state at position $l_2$ in a being attended sequence $\mathbf{y}_{l_2} \in \mathbb{R}^{D_2}$, we project it into a value vector for head $h$ according to
   $$ \mathbf{v}_{l_2, h} = \mathbf{W}_h^V \mathbf{y}_{l_2}. $$
   What is the shape of $\mathbf{W}_h^V$?

In [ ]:
# 1. The shape of W_h^Q is (D_qk, D_1).
# 2. The shape of W_h^K is (D_qk, D_2).
# 3. The shape of W_h^V is (D_v, D_2).

### P2_P2 (Day 8) (5 points, non-coding task)

For $\mathrm{M} \in\{\mathbf{Q}, \mathbf{K}, \mathbf{V}\}$, We concatenate M-projection matrices $\left\{\mathbf{W}_{h}^{\mathrm{M}}: h \in\{0,1, \cdots, H-1\}\right\}$ along axis 0 as
$$
\mathbf{W}^{\mathrm{M}}=\left[\begin{array}{c}
\mathbf{W}_{0}^{\mathrm{M}} \\
\mathbf{W}_{1}^{\mathrm{M}} \\
\vdots \\
\mathbf{W}_{H-1}^{\mathrm{M}}
\end{array}\right] .
$$
At each position $l_{1}$ in an attending sequence, we concatenate queries $\left\{\mathbf{q}_{l_{1}, h}: h \in\{0,1, \cdots, H-1\}\right\}$ along axis 0 to get
$$
\mathbf{q}_{l_{1}}=\left[\begin{array}{c}
\mathbf{q}_{l_{1}, 0} \\
\mathbf{q}_{l_{1}, 1} \\
\vdots \\
\mathbf{q}_{l_{1}, H-1}
\end{array}\right] .
$$
At each position $l_{2}$ in a being attended sequence, we concatenate keys/values $\mathbf{m} \in\{\mathbf{k}, \mathbf{v}\}$ $\left\{\mathbf{m}_{l_{2}, h}: h \in\{0,1, \cdots, H-1\}\right\}$ along axis 0 to get
$$
\mathbf{m}_{l_{2}}=\left[\begin{array}{c}
\mathbf{m}_{l_{2}, 0} \\
\mathbf{m}_{l_{2}, 1} \\
\vdots \\
\mathbf{m}_{l_{2}, H-1}
\end{array}\right] .
$$

**Do the following tasks (Reasoning is not required).**

1. What is the shape of $\mathbf{W}^{\mathrm{M}}$ for $\mathrm{M} \in\{\mathbf{Q}, \mathbf{K}, \mathbf{V}\}$ ?

2. What is the shape of $\mathbf{q}_{l_{1}}$ ?

3. What is the relationship between $\mathbf{q}_{l_{1}}$ and $\mathbf{W}^{\mathbf{Q}}$ ?

4. For $\mathbf{m} \in\{\mathbf{k}, \mathbf{v}\}$, what is the shape of $\mathbf{m}_{l_{2}}$ ?

5. What is the relationship between $\mathbf{m}_{l_{2}}$ and $\mathbf{W}^{\mathrm{M}}$ ?

1. The shape of $\mathbf{W}^{\mathbf{Q}}$ is $(H \cdot D_{qk}, D_1)$.

   The shape of $\mathbf{W}^{\mathbf{K}}$ is $(H \cdot D_{qk}, D_2)$.

   The shape of $\mathbf{W}_h^{\mathbf{V}}$ is $(H \cdot D_v, D_2)$.

2. The shape of $\mathbf{q}_{l_1}$ is $(H \cdot D_{qk}, 1)$.

3. 
   $$ \mathbf{q}_{l_1} = \mathbf{W}^{\mathbf{Q}}\mathbf{x}_{l_1}. $$

4. The shape of $\mathbf{k}_{l_2}$ is $(H \cdot D_{qk}, 1)$.

   The shape of $\mathbf{v}_{l_2}$ is $(H \cdot D_v, 1)$.

5. 
   $$ \mathbf{k}_{l_2} = \mathbf{W}^{\mathbf{K}}\mathbf{y}_{l_2}. $$
   $$ \mathbf{v}_{l_2} = \mathbf{W}^{\mathbf{V}}\mathbf{y}_{l_2}. $$

### P2_P3 (Day 8) (10 points, non-coding task)

Define function $\text{Softmax} : \mathbb{R}^d \to \mathbb{R}^d$, with the $i$-th output value as
$$
\text{Softmax}_i (\mathbf{z}) = \frac{\exp (z_i)}{\sum_{j=0}^{d-1} \exp (z_j)}.
$$
At position $l_1$ in the attending sequence, its attention score to position $l_2$ in the being attended sequence for head $h$ is denoted as $\alpha_{h,l_1l_2}$.

We can write $\alpha_{h,l_1l_2}$ in the following form:
$$
\alpha_{h,l_1l_2} = \text{Softmax}_{l_2} (\text{[???]}) ,
$$
**What is the formula in the above red box (reasoning is not required)?**

$$
\alpha_{h,l_1l_2} = \text{Softmax}_{l_2} \left( \frac{\mathbf{q}_{h,l_1}^\top \mathbf{K}_h^\top}{\sqrt{D_{qk}}} \right) ,
$$
where
$$
\mathbf{K}_h = \begin{bmatrix}
\mathbf{k}_{h,0}^\top \\
\mathbf{k}_{h,1}^\top \\
\vdots \\
\mathbf{k}_{h,L_2-1}^\top
\end{bmatrix} \in \mathbb{R}^{L_2 \times D_{qk}}.
$$